### Building LLM-Powered Applications on Databricks

By the time students reach this topic, they understand prompts and advanced prompting techniques. Now it’s time to move from prompts to applications—projects that feel usable, interactive, and capable of supporting real business tasks.

Databricks is a particularly powerful platform for this because the entire LLM workflow—data, code, models, serving, notebooks—lives in one place. No need to wire up external APIs or maintain scattered infrastructure. Students can build, test, and deploy LLM apps in the same environment.

In Topic 3, the goal is simple:
Teach students how to convert an idea into an end-to-end LLM application using Databricks models and LangChain’s LCEL.

The rest of this lesson walks them through that journey.

### 1. Setting Up the LLM Client

Most LLM apps begin by picking a model. Databricks offers several first-class options:

- Meta Llama 3 for instruction-following
- GPT and Genimi Models

In code, all of these can be used through the same LangChain interface.

In [0]:
%pip install --upgrade langchain langchain-community
dbutils.library.restartPython()

In [0]:
from langchain_community.chat_models import ChatDatabricks

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    max_tokens=300
)


This step is important for students: it shows that connecting to an enterprise model is just one line of configuration.

### 2. The First Real Application: A Simple LCEL Assistant

When students open an LLM app, there should be a clear pipeline from input → logic → output. LCEL makes this beautiful and readable.

In [0]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "You are a friendly assistant. Answer the question clearly:\n{question}"
)

assistant = prompt | llm
response = assistant.invoke({"question": "What is Delta Lake?"})
print(response)


This looks tiny, but it teaches something huge:
students now understand that an application is just the composition of components—templates, chains, and a model.

This pattern repeats again and again in any LLM system.

### 3. Building a Chat Interface (Notebook or Streamlit)

The next step is to make the assistant interactive. Even something as simple as looping through user messages creates a “chat-like” experience.

In [0]:
from langchain_core.messages import HumanMessage, AIMessage

history = []

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    history.append(HumanMessage(content=user_input))

    answer = llm.invoke(history)
    print("Assistant:", answer.content)

    history.append(AIMessage(content=answer.content))


Students immediately feel the difference:
they’re no longer testing a function—they’re building a conversational tool.

In a Databricks notebook, this interaction works beautifully. If they want a UI, Streamlit on Databricks gives them a lightweight front-end.

### 4. Summarization: One of the Easiest LLM Apps to Build

Most real LLM systems start with summarizing long text: policy documents, emails, customer reviews, operational logs.

A summarizer is also the perfect beginner application because the input and output are completely predictable.

In [0]:
from langchain_core.prompts import PromptTemplate
from langchain_community.chat_models import ChatDatabricks

# Initialize LLM
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct")

# Input text
long_text = """
Delta Lake is an open-source storage layer that brings reliability to data lakes.
It provides ACID transactions, scalable metadata handling, and unifies batch and streaming data processing.
Delta Lake runs on top of existing data lake systems such as S3, ADLS, or GCS and is fully compatible with Apache Spark APIs.
"""

# Summary prompt
summarizer_prompt = PromptTemplate.from_template("""
Summarize the following in 1-1 sentences, focusing on clarity:
{text}
""")

# LCEL chain
summarizer = summarizer_prompt | llm

# Invoke
summary = summarizer.invoke({"text": long_text})
print(summary.content)


Students immediately see the value:
this tiny app can be wrapped into a function, built into an API, or embedded into a larger agentic pipeline.

### 5. Extraction: Turning Unstructured Text Into Structured Data

Extraction apps are extremely common in enterprises—pulling names, dates, products, policies, or KPIs from text.

The difference from summarization is that extraction must be structured, often in JSON.


In [0]:
from langchain_core.prompts import PromptTemplate
from langchain_community.chat_models import ChatDatabricks

# Initialize LLM
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct")

# Define your customer text
customer_message = """
Hi, my order arrived late and the packaging was damaged.
I'm really disappointed. My name is Sarah Thompson and I expect this to be fixed.
"""

# Create extraction prompt
extract_prompt = PromptTemplate.from_template("""
Extract the following fields from the text. 
Return ONLY valid JSON.

Fields:
- issue
- customer_name
- sentiment

Text: {text}
""")

# Create chain
extractor = extract_prompt | llm

# Run extraction
response = extractor.invoke({"text": customer_message})

print(response.content)


This introduces students to the idea that LLMs can convert unstructured text into structured data, which integrates easily with Delta tables.

### 6. Classification: Turning Language Into Categories

Classification apps are useful for labeling:

sentiments
intent types
priority
severity

routing categories

A simple classification prompt looks like this:

In [0]:
classifier_prompt = PromptTemplate.from_template("""
Classify the message into one of the categories: {labels}

Message: {text}

Return only the category name.
""")

classifier = classifier_prompt | llm
print(classifier.invoke({
    "text": "You gave me amazing serivce.",
    "labels": "Complaint, Query, Appreciation"
}))


This teaches students that LLMs are not limited to writing—they can act like dynamic classifiers for downstream analytics.

### 7. SQL + LLM Apps with Unity Catalog

One of the most impressive things students can learn is that LLMs can produce SQL based on natural language—and Databricks provides both the data and the model in the same environment.

Here’s a simple NL→SQL assistant:

In [0]:
sql_prompt = PromptTemplate.from_template("""
You are a SQL generator for Databricks.

User question:
{question}

Translate it into SQL that runs on Unity Catalog tables.
""")

sql_chain = sql_prompt | llm
sql_query = sql_chain.invoke({"question": "Show top 10 customers by revenue"})
print(sql_query)


And to execute:

spark.sql(sql_query).show()


This moment is magical for students—the first time they watch an LLM write SQL and Spark execute it instantly.

This is the seed of AI/BI Genie, data copilots, and enterprise decision systems.

### 8. Bringing It Together: Small but Complete LLM Apps

By the end of Topic 3, students have built several standalone applications:

- 1 conversational chat assistant
- 2 text summarizer
- 3 structured extractor
- 4 classifier
- 5 SQL-generating data assistant

Each one is only a few lines of code, yet each represents a real pattern used in production Databricks LLM systems.

This is the point where we realize something powerful:

**“LLM applications aren’t huge systems.
They are small, clear ideas turned into chains.”**

This mindset prepares them perfectly for the next topic—agentic systems—where multiple chains start working together, and the LLM gains the ability to reason, decide, and use tools.